# M4 §1 — Elo Derivation (by hand)

Everything below is derived, not copied. Delimiters: \( \) inline math.

## 1. The logistic assumption

We need \(P(A\ \text{beats}\ B)\) to depend only on the rating difference \(d = R_A - R_B\),
to be strictly increasing in \(d\), to satisfy \(P \to 0\) as \(d \to -\infty\) and \(P \to 1\) as \(d \to +\infty\),
and to be transitive-consistent (\(A \succ B \succ C\) ratings order their win probabilities).
The canonical choice is the logistic / **Bradley–Terry** model:

\[ P(A\ \text{beats}\ B) = \sigma(\beta d) = \frac{1}{1 + e^{-\beta d}} \]

## 2. Deriving the Elo expected-score formula (the 400 convention)

Elo expresses strength differences in **log10** units. Set \(x = \beta d / \ln 10\), i.e.
switch the logistic's exponent to base 10:

\[ \sigma(\beta d) = \frac{1}{1 + 10^{-\beta d / \ln 10}} \]

Elo's convention fixes the scale by declaring: **a 400-point advantage corresponds to a
base-10 exponent of 1** (10:1 odds). That means

\[ \frac{\beta \cdot 400}{\ln 10} = 1 \quad\Longrightarrow\quad \beta = \frac{\ln 10}{400} \approx 0.005756 \]

Substituting back gives the familiar Elo expected score for team A:

\[ E_A = \frac{1}{1+10^{(R_B - R_A)/400}} \]

Check the 400-anchor: \(d = 400 \Rightarrow E_A = \frac{1}{1+10^{-1}} = \frac{10}{11} \approx 0.909\).

## 3. Worked example with real numbers

\(R_A = 1650\) (NAVI), \(R_B = 1750\) (opponent):

\[ E_A = \frac{1}{1+10^{(1750-1650)/400}} = \frac{1}{1+10^{0.25}} = \frac{1}{1+1.77828} \approx 0.359935 \]

NAVI wins (\(S = 1\)), \(K = 32\):

\[ R_A' = R_A + K(S - E_A) = 1650 + 32(1 - 0.359935) = 1650 + 20.482 \approx 1670.48 \]

## 4. The update rule is one gradient step on log-loss

Single-match log-loss for team A:

\[ \mathcal{L}(R_A) = -\big[\, S \ln E + (1-S)\ln(1-E) \,\big], \qquad E = \sigma(\beta (R_A - R_B)) \]

Derivative (chain rule, \(\sigma' = \sigma(1-\sigma)\)):

\[ \frac{d\mathcal{L}}{dR_A} = -\frac{S - E}{E(1-E)} \cdot E(1-E) \cdot \beta = -(S - E)\,\beta \]

Gradient **ascent** on log-likelihood (equivalently descent on loss) with learning rate \(\eta\):

\[ R_A' = R_A - \eta \frac{d\mathcal{L}}{dR_A} = R_A + \eta\,\beta\,(S - E) \]

Setting this equal to the Elo update \(R_A + K(S-E)\) gives the exact equivalence:

\[ K = \eta\,\beta = \eta\,\frac{\ln 10}{400} \quad\Longleftrightarrow\quad \eta = \frac{400\,K}{\ln 10} \approx 173.7\,K \]

**Elo is one SGD step on logistic log-loss with learning rate \(\eta = 173.7K\).** That is the
sentence that makes you a modeler, not a copier.

In [1]:
# 5. K as learning rate: rating trajectory toward true strength 1700
# (starts at 1500, 200 games vs average opposition of true strength 1700)
import matplotlib
import numpy as np

matplotlib.use("Agg")
import matplotlib.pyplot as plt


def expected_score(ra, rb):
    return 1.0 / (1.0 + 10.0 ** ((rb - ra) / 400.0))


rng = np.random.default_rng(7)
TRUE = 1700.0
fig, ax = plt.subplots(figsize=(9, 5))
for k in (8, 16, 32, 64):
    r = 1500.0
    path = [r]
    for _ in range(200):
        e = expected_score(r, TRUE)  # opposition at true strength
        s = 1.0 if rng.random() < e else 0.0
        r = r + k * (s - e)
        path.append(r)
    ax.plot(path, label=f"K={k}")
ax.axhline(TRUE, color="white", ls="--", lw=1, alpha=0.6)
ax.set_xlabel("games played")
ax.set_ylabel("rating")
ax.set_title("K is the learning rate: convergence vs stability trade-off")
ax.legend()
fig.tight_layout()
fig.savefig("outputs/fig_elo_k_trajectories.png", dpi=120)
plt.close(fig)
print("saved outputs/fig_elo_k_trajectories.png")
# K=8: slow, stable late; K=64: fast convergence but noisy plateau (overshoots).

saved outputs/fig_elo_k_trajectories.png


## 6. Scale equivalence — Elo is Bradley–Terry on a log10 scale

Elo curve: \(\sigma\!\big(\tfrac{\Delta R}{400}\ln 10\big)\). The Bayes/PyMC convention (M4 §4)
writes \(P = \sigma(\Delta s / 173)\) with \(s\) in Elo points:

\[ \frac{\ln 10}{400} = \frac{2.302585}{400} = 0.005756 \quad\Longrightarrow\quad \frac{1}{0.005756} = \frac{400}{\ln 10} \approx 173.7178 \]

So \( \sigma(x)\) with \(x = (\Delta R/400)\ln 10\) **is** \(\sigma(\Delta R / 173.7)\) — same curve,
different units. Sentence to keep: *"Elo is Bradley–Terry on a log10 scale; the 400-point
convention and the 173 logit-scale constant are the same thing."*